# 02 · Data Understanding : EDA (analyse exploratoire)
**Projet : E-Commerce Intelligence Platform** · Dataset Olist Brazilian E-Commerce

Contenu : statistiques descriptives, GroupBy, relations entre variables (quanti/quanti, quanti/quali, quali/quali) avec tests statistiques, visualisations et SQL.

> Exécuter les cellules **dans l'ordre** (Run All). Les cellules 3 à 12 dépendent de la table `eda` créée à la cellule 2.

## 1. Imports, chargement et conversion des types

In [ ]:
import warnings
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

path = "/kaggle/input/datasets/olistbr/brazilian-ecommerce/"
customers   = pd.read_csv(path + "olist_customers_dataset.csv")
geolocation = pd.read_csv(path + "olist_geolocation_dataset.csv")
order_items = pd.read_csv(path + "olist_order_items_dataset.csv")
payments    = pd.read_csv(path + "olist_order_payments_dataset.csv")
reviews     = pd.read_csv(path + "olist_order_reviews_dataset.csv")
orders      = pd.read_csv(path + "olist_orders_dataset.csv")
products    = pd.read_csv(path + "olist_products_dataset.csv")
sellers     = pd.read_csv(path + "olist_sellers_dataset.csv")
category    = pd.read_csv(path + "product_category_name_translation.csv")

# Conversion des dates
for c in ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
          "order_delivered_customer_date", "order_estimated_delivery_date"]:
    orders[c] = pd.to_datetime(orders[c], errors="coerce")
order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"], errors="coerce")
reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"], errors="coerce")
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"], errors="coerce")

print("Tables chargées :", 9)
print("Période :", orders["order_purchase_timestamp"].min(), "->", orders["order_purchase_timestamp"].max())

## 2. Table d'analyse (une ligne par commande livrée)
Table de travail **construite sur des copies** : les tables brutes ne sont pas modifiées. Les choix (avis le plus récent, premier article, paiement principal) sont des choix d'exploration ; les choix définitifs se feront en phase 3 (Data Preparation).

In [ ]:
DAY = 86400

# --- Agrégations au niveau commande ---
items_agg = order_items.groupby("order_id").agg(
    nb_items=("order_item_id", "count"),
    nb_sellers=("seller_id", "nunique"),
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum"),
).reset_index()

prod = products.copy()
prod["product_weight_g"] = prod["product_weight_g"].replace(0, np.nan)   # poids nul = manquant
prod["volume_cm3"] = prod["product_length_cm"] * prod["product_height_cm"] * prod["product_width_cm"]
tmp = order_items[["order_id", "product_id"]].merge(
    prod[["product_id", "product_weight_g", "volume_cm3"]], on="product_id", how="left")
g = tmp.groupby("order_id")
prod_agg = pd.DataFrame({
    "total_weight_g": g["product_weight_g"].sum(min_count=1),
    "total_volume_cm3": g["volume_cm3"].sum(min_count=1),
}).reset_index()

# --- Premier article : vendeur et catégorie ---
first_item = (order_items.sort_values(["order_id", "order_item_id"])
              .drop_duplicates("order_id")[["order_id", "seller_id", "product_id"]]
              .merge(sellers[["seller_id", "seller_state"]], on="seller_id", how="left")
              .merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
              .merge(category, on="product_category_name", how="left"))
first_item["categorie"] = first_item["product_category_name_english"].fillna("unknown")
first_item = first_item[["order_id", "seller_id", "seller_state", "categorie"]]

# --- Paiement principal, avis le plus récent, client ---
pay_main = (payments.sort_values("payment_value", ascending=False)
            .drop_duplicates("order_id")[["order_id", "payment_type"]])
rev = (reviews.sort_values("review_answer_timestamp")
       .drop_duplicates("order_id", keep="last")[["order_id", "review_score"]])
cust = customers[["customer_id", "customer_unique_id", "customer_state"]]

# --- Commandes livrées avec date de livraison ---
eda = orders[(orders["order_status"] == "delivered")
             & orders["order_delivered_customer_date"].notna()].copy()

eda["delivery_days"] = (eda["order_delivered_customer_date"] - eda["order_purchase_timestamp"]).dt.total_seconds() / DAY
eda["promised_days"] = (eda["order_estimated_delivery_date"] - eda["order_purchase_timestamp"]).dt.total_seconds() / DAY
eda["gap_days"]      = eda["promised_days"] - eda["delivery_days"]          # > 0 : en avance
eda["seller_days"]   = (eda["order_delivered_carrier_date"] - eda["order_purchase_timestamp"]).dt.total_seconds() / DAY
eda["carrier_days"]  = (eda["order_delivered_customer_date"] - eda["order_delivered_carrier_date"]).dt.total_seconds() / DAY
eda["is_late"]       = (eda["order_delivered_customer_date"] > eda["order_estimated_delivery_date"]).astype(int)

eda["purchase_month"] = eda["order_purchase_timestamp"].dt.strftime("%Y-%m")
eda["purchase_dow"]   = eda["order_purchase_timestamp"].dt.dayofweek
eda["purchase_hour"]  = eda["order_purchase_timestamp"].dt.hour
eda["is_weekend"]     = (eda["purchase_dow"] >= 5).astype(int)

eda = (eda.merge(cust, on="customer_id", how="left")
          .merge(items_agg, on="order_id", how="left")
          .merge(prod_agg, on="order_id", how="left")
          .merge(first_item, on="order_id", how="left")
          .merge(pay_main, on="order_id", how="left")
          .merge(rev, on="order_id", how="left"))

eda["same_state"] = np.where(eda["seller_state"].isna(), np.nan,
                             (eda["seller_state"] == eda["customer_state"]).astype(float))

print("Table d'analyse :", eda.shape)
print("Commandes livrées exploitables :", len(eda))
print("Taux de retard (%) :", round(eda["is_late"].mean() * 100, 2))
print("Même état vendeur/client (%) :", round(eda["same_state"].mean() * 100, 1))

## 3. Statistiques descriptives

In [ ]:
quanti = ["delivery_days", "promised_days", "gap_days", "seller_days", "carrier_days",
          "total_price", "total_freight", "nb_items", "total_weight_g",
          "total_volume_cm3", "review_score"]

desc = eda[quanti].describe(percentiles=[.25, .5, .75, .95]).T
desc["skew"] = eda[quanti].skew()
desc["kurtosis"] = eda[quanti].kurtosis()
desc["manquantes_%"] = eda[quanti].isna().mean() * 100
print("=== Variables quantitatives ===")
display(desc)

print("\n=== Variables qualitatives ===")
for c in ["customer_state", "seller_state", "categorie", "payment_type", "is_late", "same_state", "is_weekend"]:
    vc = eda[c].value_counts(normalize=True, dropna=False).head(8) * 100
    print(f"\n{c} (% , {eda[c].nunique()} modalités) :")
    print(vc.round(1).to_string())

**Conclusion (à rédiger) :** asymétrie des distributions, variables à valeurs manquantes, modalités dominantes.

## 4. Distributions des variables clés

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9))

sns.histplot(eda["delivery_days"].clip(upper=60), bins=60, ax=axes[0, 0])
axes[0, 0].set_title("Durée de livraison (jours, tronquée à 60)")

sns.histplot(eda["delivery_days"].clip(upper=60), bins=60, color="tab:blue", label="Réelle", alpha=0.6, ax=axes[0, 1])
sns.histplot(eda["promised_days"].clip(upper=60), bins=60, color="tab:orange", label="Promise", alpha=0.6, ax=axes[0, 1])
axes[0, 1].legend(); axes[0, 1].set_title("Durée réelle vs promesse d'Olist")

sns.histplot(eda["gap_days"].clip(-30, 60), bins=60, ax=axes[0, 2])
axes[0, 2].axvline(0, color="red", ls="--")
axes[0, 2].set_title("Écart promesse - réel (> 0 : en avance)")

eda[["seller_days", "carrier_days"]].clip(lower=0).median().plot.bar(ax=axes[1, 0])
axes[1, 0].set_title("Durée médiane : vendeur vs transport (jours)")

sns.histplot(eda["total_freight"].clip(upper=100), bins=50, ax=axes[1, 1])
axes[1, 1].set_title("Frais de port par commande (BRL, tronqué à 100)")

eda["review_score"].value_counts().sort_index().plot.bar(ax=axes[1, 2])
axes[1, 2].set_title("Répartition des notes d'avis")

plt.tight_layout(); plt.show()

print("MAE promesse Olist   :", round((eda["delivery_days"] - eda["promised_days"]).abs().mean(), 2), "jours")
print("MAE médiane globale  :", round((eda["delivery_days"] - eda["delivery_days"].median()).abs().mean(), 2), "jours")
print("Asymétrie (skew) de la durée :", round(eda["delivery_days"].skew(), 2))

**Conclusion (à rédiger) :** la promesse d'Olist est-elle large ? Où se perd le temps (vendeur ou transport) ?

## 5. GroupBy

In [ ]:
# 1. Par état du client
par_etat = (eda.groupby("customer_state")
               .agg(nb=("order_id", "count"), taux_retard=("is_late", "mean"),
                    duree_mediane=("delivery_days", "median"),
                    promesse_mediane=("promised_days", "median"),
                    fret_moy=("total_freight", "mean"), note_moy=("review_score", "mean"))
               .query("nb >= 200").sort_values("taux_retard", ascending=False))
print("=== Par état du client ===")
display(par_etat)

# 2. Par mois
par_mois = (eda.groupby("purchase_month")
               .agg(nb=("order_id", "count"), taux_retard=("is_late", "mean"),
                    duree_moy=("delivery_days", "mean"), ca=("total_price", "sum"))
               .query("nb >= 500"))
print("\n=== Par mois ===")
display(par_mois)

# 3. Retard et satisfaction
print("\n=== Note d'avis selon le retard ===")
display(eda.groupby("is_late")["review_score"].agg(["count", "mean", "median"]))

# 4. Même état ou non
print("\n=== Même état vendeur/client ===")
display(eda.groupby("same_state").agg(nb=("order_id", "count"), duree_mediane=("delivery_days", "median"),
                                      taux_retard=("is_late", "mean"), fret_moy=("total_freight", "mean")))

# 5. Par catégorie (premier article)
par_cat = (eda.groupby("categorie")
              .agg(nb=("order_id", "count"), note=("review_score", "mean"),
                   taux_retard=("is_late", "mean"), duree_mediane=("delivery_days", "median"))
              .query("nb >= 300").sort_values("note"))
print("\n=== Catégories les moins bien notées ===")
display(par_cat.head(10))

# 6. Par mode de paiement
print("\n=== Par mode de paiement ===")
display(eda.groupby("payment_type").agg(nb=("order_id", "count"), panier_moy=("total_price", "mean"),
                                        taux_retard=("is_late", "mean")))

# 7. Clients : achats répétés et montant dépensé
oc = eda.groupby("customer_unique_id").agg(nb_cmd=("order_id", "nunique"), depense=("total_price", "sum"))
print("\n=== Clients ===")
print("Clients uniques :", len(oc))
print("Taux d'achat répété (%) :", round((oc["nb_cmd"] > 1).mean() * 100, 2))
print(oc["nb_cmd"].value_counts().sort_index().head(5).to_string())
display(oc["depense"].describe())

**Conclusion (à rédiger) :** états les plus touchés par les retards, mois critiques, catégories les moins bien notées, faible taux d'achat répété.

## 6. Relations quanti vs quanti (Spearman)

In [ ]:
vars_qq = ["delivery_days", "promised_days", "total_price", "total_freight",
           "nb_items", "total_weight_g", "total_volume_cm3", "review_score"]

rows = []
for i, a in enumerate(vars_qq):
    for b in vars_qq[i + 1:]:
        d = eda[[a, b]].dropna()
        rho, p = stats.spearmanr(d[a], d[b])
        rows.append([a, b, len(d), rho, p])
qq = pd.DataFrame(rows, columns=["var1", "var2", "n", "rho", "p"])
qq["force"] = pd.cut(qq["rho"].abs(), [0, 0.1, 0.3, 0.5, 1.01],
                     labels=["négligeable", "faible", "moyenne", "forte"], include_lowest=True)
display(qq.reindex(qq["rho"].abs().sort_values(ascending=False).index).reset_index(drop=True))

plt.figure(figsize=(9, 7))
sns.heatmap(eda[vars_qq].corr(method="spearman"), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Quanti vs quanti : corrélations de Spearman")
plt.show()

## 7. Relations quanti vs quali (Mann-Whitney, Kruskal-Wallis)

In [ ]:
def mannwhitney(df, num, binaire):
    a = df.loc[df[binaire] == 1, num].dropna()
    b = df.loc[df[binaire] == 0, num].dropna()
    u, p = stats.mannwhitneyu(a, b, alternative="two-sided")
    effet = 2 * u / (len(a) * len(b)) - 1            # corrélation bisériale par rang
    return [num, binaire, round(a.median(), 2), round(b.median(), 2), p, round(effet, 3)]

def kruskal(df, num, cat, min_n=100):
    gs = [x[num].dropna().values for _, x in df.groupby(cat) if x[num].notna().sum() >= min_n]
    h, p = stats.kruskal(*gs)
    n = sum(len(x) for x in gs)
    return [num, cat, len(gs), p, round(h * (n + 1) / (n ** 2 - 1), 3)]   # epsilon²

# Variables binaires : Mann-Whitney
res_bin = []
for b in ["is_late", "same_state", "is_weekend"]:
    for n in ["total_freight", "total_weight_g", "total_price", "promised_days", "review_score"]:
        if n == "review_score" and b != "is_late":
            continue
        res_bin.append(mannwhitney(eda, n, b))
print("=== Quanti vs binaire (Mann-Whitney) ===")
display(pd.DataFrame(res_bin, columns=["quanti", "binaire", "médiane_si_1", "médiane_si_0", "p", "effet"]))

# Variables à plusieurs modalités : Kruskal-Wallis
res_cat = [kruskal(eda, n, c)
           for c in ["customer_state", "seller_state", "categorie", "payment_type", "purchase_dow"]
           for n in ["delivery_days", "total_freight"]]
print("\n=== Quanti vs quali (Kruskal-Wallis) ===")
display(pd.DataFrame(res_cat, columns=["quanti", "quali", "nb_groupes", "p", "epsilon²"]))

## 8. Relations quali vs quali (chi² et V de Cramér)
*Cette cellule peut prendre quelques dizaines de secondes.*

In [ ]:
def cramers_v(x, y, min_count=100):
    d = pd.DataFrame({"x": x.astype(str), "y": y.astype(str)})
    d = d[~d.isin(["nan"]).any(axis=1)]
    for col in ["x", "y"]:
        vc = d[col].value_counts()
        d[col] = d[col].where(d[col].isin(vc[vc >= min_count].index), "autre")
    ct = pd.crosstab(d["x"], d["y"])
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    return np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1))), p

quali = ["is_late", "customer_state", "seller_state", "categorie",
         "payment_type", "same_state", "purchase_dow"]

rows = []
for i, a in enumerate(quali):
    for b in quali[i + 1:]:
        v, p = cramers_v(eda[a], eda[b])
        rows.append([a, b, p, v])
qa = pd.DataFrame(rows, columns=["var1", "var2", "p", "V_Cramer"])
display(qa.sort_values("V_Cramer", ascending=False).reset_index(drop=True))

m = pd.DataFrame(np.eye(len(quali)), index=quali, columns=quali)
for a, b, _, v in rows:
    m.loc[a, b] = m.loc[b, a] = v
plt.figure(figsize=(8, 6))
sns.heatmap(m, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1)
plt.title("Quali vs quali : V de Cramér")
plt.show()

print("Retard (%) selon le mode de paiement :")
display((pd.crosstab(eda["payment_type"], eda["is_late"], normalize="index") * 100).round(1))

## 9. Tests statistiques ciblés
> Avec ~96 000 lignes, la p-value est presque toujours minuscule : **interpréter la taille d'effet** (≈ 0,1 faible, 0,3 moyen, 0,5 fort).

In [ ]:
alpha = 0.05
verdict = lambda p: "significatif" if p < alpha else "non significatif"

# 0. Normalité de la durée de livraison (échantillon de 5000)
s = eda["delivery_days"].dropna().sample(5000, random_state=42)
p = stats.shapiro(s).pvalue
print(f"[Normalité] Shapiro, durée : p={p:.2e} -> non normale : tests non paramétriques\n")

# 1. Retard et note d'avis : Mann-Whitney
a = eda.loc[eda["is_late"] == 1, "review_score"].dropna()
b = eda.loc[eda["is_late"] == 0, "review_score"].dropna()
u, p = stats.mannwhitneyu(a, b, alternative="two-sided")
print(f"[Retard -> note] Mann-Whitney p={p:.2e} ({verdict(p)}), effet={2*u/(len(a)*len(b))-1:.2f}")
print(f"   note moyenne en retard={a.mean():.2f} | à l'heure={b.mean():.2f}\n")

# 2. Retard et état du client : chi² + V de Cramér
ct = pd.crosstab(eda["customer_state"], eda["is_late"])
chi2, p, dof, _ = stats.chi2_contingency(ct)
v = np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))
print(f"[Retard x état] chi² p={p:.2e} ({verdict(p)}), V de Cramér={v:.2f}\n")

# 3. Durée selon l'état : Kruskal-Wallis + epsilon²
gs = [x["delivery_days"].values for _, x in eda.groupby("customer_state") if len(x) >= 30]
h, p = stats.kruskal(*gs)
n = sum(len(x) for x in gs)
print(f"[Durée x état] Kruskal-Wallis p={p:.2e} ({verdict(p)}), epsilon²={h*(n+1)/(n**2-1):.2f}\n")

# 4. Durée selon même état / états différents
a = eda.loc[eda["same_state"] == 1, "delivery_days"]
b = eda.loc[eda["same_state"] == 0, "delivery_days"]
u, p = stats.mannwhitneyu(a, b, alternative="two-sided")
print(f"[Durée x même état] Mann-Whitney p={p:.2e} ({verdict(p)}), effet={2*u/(len(a)*len(b))-1:.2f}")
print(f"   médiane même état={a.median():.1f} j | états différents={b.median():.1f} j\n")

# 5. Durée et variables numériques : Spearman
for c in ["total_freight", "total_weight_g", "total_volume_cm3", "total_price", "nb_items"]:
    d = eda[["delivery_days", c]].dropna()
    r, p = stats.spearmanr(d["delivery_days"], d[c])
    print(f"[Durée ~ {c:17s}] rho={r:5.2f}, p={p:.1e}")

print("\nRappel : avec ~96 000 lignes, la p-value est presque toujours minuscule. Interprète la taille d'effet.")

## 10. Visualisations des relations

In [ ]:
# 1. Retard et durée par état
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
(par_etat["taux_retard"] * 100).sort_values(ascending=False).plot.bar(ax=ax[0], title="Taux de retard (%) par état du client")
par_etat["duree_mediane"].sort_values(ascending=False).plot.bar(ax=ax[1], title="Durée médiane (jours) par état du client")
plt.tight_layout(); plt.show()

# 2. Évolution mensuelle
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
(par_mois["taux_retard"] * 100).plot(marker="o", ax=ax[0], title="Taux de retard (%) par mois")
par_mois["nb"].plot(marker="o", ax=ax[1], color="tab:green", title="Commandes livrées par mois")
par_mois["ca"].plot(marker="o", ax=ax[2], color="tab:red", title="Chiffre d'affaires par mois (BRL)")
for a in ax: a.tick_params(axis="x", rotation=60)
plt.tight_layout(); plt.show()

# 3. Retard et satisfaction
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
(pd.crosstab(eda["is_late"], eda["review_score"], normalize="index") * 100).plot.bar(
    stacked=True, ax=ax[0], title="Notes (%) : à l'heure (0) vs en retard (1)")
sns.boxplot(data=eda, x="is_late", y="review_score", ax=ax[1])
ax[1].set_title("Note d'avis selon le retard")
plt.tight_layout(); plt.show()

# 4. Fret, poids et durée
ech = eda.sample(8000, random_state=42)
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
sns.scatterplot(data=ech, x="total_freight", y="delivery_days", alpha=0.3, ax=ax[0])
ax[0].set_xlim(0, 150); ax[0].set_ylim(0, 80); ax[0].set_title("Fret vs durée de livraison")
sns.scatterplot(data=ech, x="total_weight_g", y="delivery_days", alpha=0.3, ax=ax[1])
ax[1].set_xlim(0, 20000); ax[1].set_ylim(0, 80); ax[1].set_title("Poids vs durée de livraison")
plt.tight_layout(); plt.show()

# 5. Même état ou non
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
sns.boxplot(data=eda, x="same_state", y=eda["delivery_days"].clip(upper=60), ax=ax[0])
ax[0].set_title("Durée selon : même état (1) ou non (0)")
sns.boxplot(data=eda, x="same_state", y=eda["total_freight"].clip(upper=100), ax=ax[1])
ax[1].set_title("Frais de port selon : même état (1) ou non (0)")
plt.tight_layout(); plt.show()

# 6. Catégories et clients
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
par_cat["note"].head(12).plot.barh(ax=ax[0], title="Catégories les moins bien notées")
sns.histplot(oc["depense"].clip(upper=1000), bins=60, ax=ax[1])
ax[1].set_title("Montant dépensé par client (tronqué à 1000 BRL)")
plt.tight_layout(); plt.show()

# 7. Jour et heure d'achat
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
eda.groupby("purchase_dow")["is_late"].mean().mul(100).plot.bar(ax=ax[0], title="Taux de retard (%) selon le jour d'achat (0 = lundi)")
eda.groupby("purchase_hour")["order_id"].count().plot(ax=ax[1], marker="o", title="Commandes selon l'heure d'achat")
plt.tight_layout(); plt.show()

**Conclusion (à rédiger) :** une phrase de constat sous chaque graphique.

## 11. SQL (SQLite en mémoire)
Jointures, `GROUP BY` / `HAVING`, CTE et fonctions de fenêtre (`LAG`, `RANK`).

In [ ]:
con = sqlite3.connect(":memory:")

eda[["order_id", "customer_unique_id", "customer_state", "seller_state", "categorie",
     "purchase_month", "delivery_days", "promised_days", "is_late", "total_price",
     "total_freight", "review_score"]].to_sql("eda", con, index=False, if_exists="replace")
order_items[["order_id", "order_item_id", "product_id", "seller_id", "price"]].to_sql(
    "order_items", con, index=False, if_exists="replace")

def q(sql):
    return pd.read_sql_query(sql, con)

# 1. GROUP BY + HAVING : retard et durée par état
print("--- 1. Retard et durée par état ---")
display(q("""
SELECT customer_state, COUNT(*) AS nb,
       ROUND(AVG(is_late) * 100, 2) AS taux_retard_pct,
       ROUND(AVG(delivery_days), 1) AS duree_moy
FROM eda
GROUP BY customer_state
HAVING nb >= 200
ORDER BY taux_retard_pct DESC
LIMIT 10;
"""))

# 2. JOIN : vendeurs avec le plus fort taux de retard
print("--- 2. Vendeurs à fort taux de retard (>= 30 commandes) ---")
display(q("""
SELECT oi.seller_id, COUNT(DISTINCT e.order_id) AS nb,
       ROUND(AVG(e.is_late) * 100, 1) AS taux_retard_pct
FROM order_items oi
JOIN eda e ON e.order_id = oi.order_id
GROUP BY oi.seller_id
HAVING nb >= 30
ORDER BY taux_retard_pct DESC
LIMIT 10;
"""))

# 3. CTE : note moyenne et retard par catégorie
print("--- 3. Catégories les moins bien notées ---")
display(q("""
WITH cat AS (
  SELECT categorie, review_score, is_late
  FROM eda
)
SELECT categorie, COUNT(*) AS nb,
       ROUND(AVG(review_score), 2) AS note_moy,
       ROUND(AVG(is_late) * 100, 1) AS taux_retard_pct
FROM cat
GROUP BY categorie
HAVING nb >= 300
ORDER BY note_moy ASC
LIMIT 10;
"""))

# 4. Fonction de fenêtre LAG : évolution mensuelle du chiffre d'affaires
print("--- 4. Évolution mensuelle du CA ---")
display(q("""
WITH mensuel AS (
  SELECT purchase_month AS mois, COUNT(*) AS commandes, ROUND(SUM(total_price), 0) AS ca
  FROM eda
  GROUP BY purchase_month
  HAVING commandes >= 500
)
SELECT mois, commandes, ca,
       ROUND((ca - LAG(ca) OVER (ORDER BY mois)) * 100.0 / LAG(ca) OVER (ORDER BY mois), 1) AS variation_pct
FROM mensuel
ORDER BY mois;
"""))

# 5. Fonction de fenêtre RANK : classement des états par taux de retard
print("--- 5. Classement des états ---")
display(q("""
SELECT customer_state, nb, taux_retard_pct,
       RANK() OVER (ORDER BY taux_retard_pct DESC) AS rang
FROM (
  SELECT customer_state, COUNT(*) AS nb, ROUND(AVG(is_late) * 100, 2) AS taux_retard_pct
  FROM eda GROUP BY customer_state HAVING nb >= 200
)
ORDER BY rang
LIMIT 10;
"""))

# 6. Vérification : le résultat SQL concorde avec pandas
sql_res = q("SELECT ROUND(AVG(is_late) * 100, 2) AS t FROM eda")["t"][0]
print("Taux de retard SQL :", sql_res, "| pandas :", round(eda["is_late"].mean() * 100, 2))

## 12. Synthèse des constats

In [ ]:
top_etat = par_etat.index[0]
bot_etat = par_etat.index[-1]

print("=== SYNTHÈSE DE L'EDA ===")
print(f"- Commandes livrées exploitables : {len(eda):,}")
print(f"- Durée de livraison : médiane {eda['delivery_days'].median():.1f} j, moyenne {eda['delivery_days'].mean():.1f} j")
print(f"- Promesse d'Olist : médiane {eda['promised_days'].median():.1f} j "
      f"(avance moyenne {eda['gap_days'].mean():.1f} j)")
print(f"- Taux de retard : {eda['is_late'].mean()*100:.2f} % (classes déséquilibrées)")
print(f"- État avec le plus fort taux de retard : {top_etat} ({par_etat.loc[top_etat, 'taux_retard']*100:.1f} %)")
print(f"- État avec le plus faible taux de retard : {bot_etat} ({par_etat.loc[bot_etat, 'taux_retard']*100:.1f} %)")
print(f"- Note moyenne : {eda.loc[eda.is_late==0, 'review_score'].mean():.2f} à l'heure "
      f"contre {eda.loc[eda.is_late==1, 'review_score'].mean():.2f} en retard")
print(f"- Commandes entre états différents : {(1 - eda['same_state'].mean())*100:.1f} %")
print(f"- Clients ayant commandé plus d'une fois : {(oc['nb_cmd'] > 1).mean()*100:.2f} %")
print(f"- Catégorie la moins bien notée : {par_cat.index[0]} ({par_cat['note'].iloc[0]:.2f})")

### Points à retenir pour la phase suivante (Data Preparation)
- Variables candidates : fret, poids, volume, état client / vendeur, même état, mois et jour d'achat.
- **Fuites de données à éviter** : `gap_days`, `seller_days`, `carrier_days`, `review_score` et toute date postérieure à l'achat.
- La **distance vendeur-client** sera calculée en phase 3, après nettoyage de `geolocation`.